# Lab: Text Vectorization - From Strings to Numbers

## 1. Introduction

Computers cannot understand text directly. They only understand numbers. 

**Vectorization** is the process of converting text into numerical vectors.
In this lab, we will explore 4 methods:
1.  **One-Hot Encoding**: Simple but moemory consumable!
2.  **TF-IDF**: Counts words but weighs them by importance.
3.  **Word2Vec**: Learns "meaning" from context.
4.  **PyTorch Embeddings**: The standard for Deep Learning.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import matplotlib.pyplot as plt
import torch.optim as optim

In [ ]:
import pandas as pd

In [ ]:
# A tiny corpus to easily view the matrices

corpus = [
    "the king loves the queen",
    "the queen loves the king",
    "the apple is red and sweet",
    "the orange is a fruit",
    "fruits are sweet",
    "the king eats the apple"
]

## 2. Sparse Representations

### Method 1: One-Hot Encoding
Each word is a vector of size V (vocab size). It has a single 1 and zeros everywhere else.

#### Pandas get_dummies()

In [ ]:
print(corpus[2])
words = corpus[2].split()

# Generate one-hot encoding using pandas
one_hot_df_a = pd.get_dummies(words).astype(int)

# Displaying the result
print(f"Sentence: {words}")
display(one_hot_df_a)

#### CountVectorizer: Bag of Words (transforms text documents into a numerical matrix of word counts, acting as a "bag of words" model that tokenizes text)

In [ ]:
# Initialize CountVectorizer with binary=True for one-hot encoding (presence/absence)
vectorizer = CountVectorizer(binary=True)

# Create the one-hot encoded matrix
one_hot_matrix = vectorizer.fit_transform(corpus)

print(one_hot_matrix.shape, one_hot_matrix.toarray())

In [ ]:
# Retrieve the vocabulary (feature names)
vocab = vectorizer.get_feature_names_out()

# Display details using a DataFrame for readability
one_hot_df = pd.DataFrame(one_hot_matrix.toarray(), columns=vocab)

print("Vocabulary:", vocab)
print(one_hot_df.shape)
print("\nOne-Hot Encoded Matrix:")
display(one_hot_df)

In [ ]:
one_hot_df.columns

### Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# In one_hot_df, rows are sentences and columns are words.

apple_vec = one_hot_df[['apple']].T # Transpose to column vector
red_vec = one_hot_df[['red']].T
orange_vec = one_hot_df[['orange']].T

print(f"Similarity apple vs red: {cosine_similarity(apple_vec, red_vec)[0][0]}") 
print(f"Similarity apple vs orange: {cosine_similarity(apple_vec, orange_vec)[0][0]}")

In [ ]:
corpus

In [ ]:
king_vec = one_hot_df[['king']].T 
queen_vec = one_hot_df[['queen']].T

print(king_vec)
print(queen_vec)
    
similarity_1 = cosine_similarity(king_vec, queen_vec)[0][0]
print(f"Similarity king vs queen: {similarity_1}")

In [ ]:
print(corpus[2])
apple_vec = one_hot_df_a[['apple']].T
red_vec = one_hot_df_a[['red']].T
    
similarity_2 = cosine_similarity(apple_vec, red_vec)[0][0]
print(f"Similarity 'apple' vs 'orange': {similarity_2}")

### Method 2: TF-IDF (Term Frequency - Inverse Document Frequency)
Gives higher weight to rare words. "The" appears everywhere so it gets low weight.

In [ ]:
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(corpus)

print("Features:", tfidf_vec.get_feature_names_out())
print("\nMatrix Shape:", X_tfidf.shape)

print("\nTF-IDF Matrix (First Sentence & sixth sentence):")
print(f"Corpus- {corpus[0]}: {np.round(X_tfidf.toarray()[0], 2)}")
print(f"Corpus- {corpus[5]}: {np.round(X_tfidf.toarray()[5], 2)}")

In [ ]:
# Get feature names and find indices for 'apple' and 'fruits'
feature_names = tfidf_vec.get_feature_names_out()
print(f"Feature names: {feature_names}")

apple_idx = list(feature_names).index('apple') # Test with values
fruits_idx = list(feature_names).index('fruits') # Test with values

# Extract word vectors (columns) and transpose to shape (1, n_samples)
apple_vec = X_tfidf[:, apple_idx].T
fruits_vec = X_tfidf[:, fruits_idx].T

# Calculate cosine similarity
similarity = cosine_similarity(apple_vec, fruits_vec)[0][0]
print(f"Cosine Similarity between 'apple' and 'fruits': {similarity}")


In [ ]:
corpus

### Why is the similarity 0.0?

We might expect **"apple"** and **"fruits"** to be similar. However, in TF-IDF:
1.  **"apple"** appears in sentences 2 and 5.
2.  **"fruits"** appears ONLY in sentence 4.

They never appear in the same sentence together. Therefore, their vectors are **orthogonal** (perpendicular), and their dot product is 0.

**This is the limitation of Sparse Representations.** They cannot capture semantic similarity unless words literally appear together. 

**Solution:** Dense Embeddings (Word2Vec)!

## 3. Dense & Learned Representations (Embeddings)

Instead of `0` and `1`, we want vectors like `[0.2, -0.9, 1.4]`. These are **Dense Vectors**.

### Method 3: Word2Vec (Skip-Gram) Concept
We train a neural network to predict context words. If "King" and "Queen" both appear near "loves", their vectors will move closer together.

Let's build a tiny Word2Vec in PyTorch!

In [ ]:
# 1. Prepared Training Data (Target Word -> Context Word)

# Window size 1: "the king loves" -> (king, the), (king, loves)
data = []
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for i, w in enumerate(vocab)}

In [ ]:
print(word_to_ix)

In [ ]:
print(ix_to_word)

In [ ]:
context_window = 1 # Number of words to the left and right of the target word

for sentence in corpus:
    words = sentence.split()
    # print(words)
    for i, target in enumerate(words):

        # Get neighbors
        for j in range(max(0, i - context_window), min(len(words), i + context_window + 1)):
            if i != j:
                context = words[j]

                # Verify word exists in vocab (CountVectorizer might lowercase or strip punct)
                if target in word_to_ix and context in word_to_ix:
                    # print(target, context)
                    data.append((word_to_ix[target], word_to_ix[context]))

print(f"Training Pairs sample: {data[:5]} (Indices)")
print(data)
print(f"Vocabulary Mapping: {word_to_ix}")


In [ ]:
# 2. The Model

class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)
        
    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        out = self.linear(embeds)
        return out

# 3. Train
EMBED_DIM = 4
model = Word2Vec(len(vocab), EMBED_DIM) # embedding dimension is 4
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()

print("Training Word2Vec...")
for epoch in range(500):
    total_loss = 0

    # Creating batch tensors from list
    inputs = torch.tensor([ex[0] for ex in data])
    targets = torch.tensor([ex[1] for ex in data])
    
    optimizer.zero_grad()
    output = model(inputs)
    loss = loss_fn(output, targets)
    loss.backward()
    optimizer.step()
    
print("Training Done!")

In [ ]:
print(vocab)

### Method 4: PyTorch nn.Embedding

`nn.Embedding` is basically what we used above. 
It's a **Lookup Table**.

It stores the dense vector for each word.

Let's inspect the embeddings we just generated.

In [ ]:
# Get the weights from the embedding layer

embeddings = model.embeddings.weight.data

def get_vector(word):
    if word in word_to_ix:
        idx = word_to_ix[word]
        return embeddings[idx]
    return None

print("Vector for 'king':", get_vector("king"))
print("Vector for 'apple':", get_vector("apple"))

## 4. Semantic Similarity Demo

**Cosine Similarity** measures the angle between two vectors.

*   **1.0**: Same direction (Identical meaning assumption)
*   **0.0**: 90 degrees (Unrelated)
*   **-1.0**: Opposite direction

In [ ]:
def cosine_similarity_torch(v1, v2):
    # Cosine Sim = (A . B) / (||A|| * ||B||)
    dot_product = torch.dot(v1, v2)
    norm1 = torch.norm(v1)
    norm2 = torch.norm(v2)
    return (dot_product / (norm1 * norm2)).item()

In [ ]:
# Let's calculate similarities

pairs = [
    ("king", "queen"),
    ("apple", "orange"),
    ("king", "apple"), # Should be lower
    ("queen", "red")
]

for w1, w2 in pairs:
    v1 = get_vector(w1)
    v2 = get_vector(w2)

    sim = cosine_similarity_torch(v1, v2)
    print(f"Similarity({w1}, {w2}) = {sim:.4f}")

**Note:** On a tiny corpus like this, results might be noisy. 

Real Word2Vec is trained on BILLIONS of words. 

But you should see that King/Queen are closer than King/Apple.